# Agent Identity SDK: Manual Client Usage Example

This notebook demonstrates how to interactively use the `IdentityClient` directly without relying on decorators or `AgentIdentityContext`.

In [ ]:
import os

# Ensure you DO NOT commit these secrets to version control.
# os.environ["HUAWEICLOUD_SDK_AK"] = "your-access-key-id"
# os.environ["HUAWEICLOUD_SDK_SK"] = "your-secret-access-key"
# os.environ["HUAWEICLOUD_SDK_REGION"] = "ap-southeast-4"
# os.environ["HUAWEICLOUD_SDK_PROJECT_ID"] = "your-project-id"
# os.environ["HUAWEICLOUD_SDK_DOMAIN_ID"] = "your-domain-id"

print("Environment variables ready.")

In [ ]:
import uuid
from huaweicloudsdkagentidentity.v1 import AuthorizerType
from agentarts.sdk import IdentityClient

### Initialize Client and Create Workload

We directly instantiate the client and create our workload identity.

In [ ]:
region = os.getenv("HUAWEICLOUD_SDK_REGION", "ap-southeast-4")
client = IdentityClient(region=region, ignore_ssl_verification=True)
print(f"IdentityClient initialized for region: {region}")

random_suffix = uuid.uuid4().hex[:8]
user_id = f"user-{random_suffix}"
workload_name = f"manual-workload-{random_suffix}"

print(f"Creating workload identity: {workload_name}...")
workload = client.create_workload_identity(
    name=workload_name, authorizer_type=AuthorizerType.NONE
)
print(f"Created Workload: {workload.name}")

### Retrieve Workload Access Token

Instead of setting this in context, we will hold onto the returned string and pass it manually to subsequent calls.

In [ ]:
print(f"Getting workload access token for user {user_id}...")
workload_access_token = client.create_workload_access_token(
    workload_name=workload.name, user_id=user_id
)
print("Token successfully retrieved!")

### Fetch an API Key (Example)

Now we request a specific resource (API Key) by manually passing the `workload_access_token`.

In [ ]:
provider_name = f"manual-provider-{random_suffix}"
print(f"Ensuring API Key provider '{provider_name}' exists...")
try:
    client.create_api_key_credential_provider(
        name=provider_name, api_key="sk-dummy-key"
    )
except Exception as e:
    print(f"Provider might already exist: {e}")

print("\nManually fetching API key...")
api_key = client.get_resource_api_key(
    provider_name=provider_name,
    workload_access_token=workload_access_token
)
print(f"Successfully retrieved API Key: {api_key}")